Step 1: Project Initialization and Dependency Setup

In [1]:
!pip install groq

In [2]:
!pip install chromadb sentence-transformers

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Step 2: Data Loading and Preprocessing

In [4]:
import pandas as pd

rag_df = pd.read_csv("/content/drive/MyDrive/Log_Anamoly_Detection/results/rag_documents.csv")
documents = rag_df["RAG_Document"].tolist()

print("Total RAG documents:", len(documents))
print(documents[0])

Total RAG documents: 3378
Block ID        : blk_8462687553742484299
Label           : Fail
Anomaly Type(s) : duplicate_pattern

--- Details ---
Event Sequence  : E5 -> E5 -> E5 -> E22 -> E11 -> E9 -> E11 -> E9 -> E11 -> E9 -> E26 -> E26 -> E26 -> E23 -> E23 -> E23 -> E21 -> E20 -> E21 -> E21
Readable Form   : [*]Receiving block[*]src:[*]dest:[*] -> [*]Receiving block[*]src:[*]dest:[*] -> [*]Receiving block[*]src:[*]dest:[*] -> [*]BLOCK* NameSystem[*]allocateBlock:[*] -> [*]PacketResponder[*]for block[*]terminating[*] -> [*]Received block[*]of size[*]from[*] -> [*]PacketResponder[*]for block[*]terminating[*] -> [*]Received block[*]of size[*]from[*] -> [*]PacketResponder[*]for block[*]terminating[*] -> [*]Received block[*]of size[*]from[*] -> [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*] -> [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*] -> [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*] 

Step 3: Semantic Embedding Generation

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(documents)

print("Embedding shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape: (3378, 384)


Step 4: Vector Database Indexing with FAISS

In [6]:
import chromadb
from chromadb.config import Settings

client = chromadb.Client(
    Settings(
        persist_directory="./chroma_db"  # folder where DB will be saved
    )
)

collection = client.get_or_create_collection(name="anomaly_logs")

In [7]:
if collection.count() == 0:
    embeddings = model.encode(documents)

    collection.add(
        documents=documents,
        embeddings=embeddings.tolist(),
        ids=[str(i) for i in range(len(documents))]
    )

print("Vectors stored in ChromaDB:", collection.count())

Vectors stored in ChromaDB: 3378


Step 5: Retrieval and LLM Analysis Logic

In [8]:
def retrieve_similar_anomalies(query, k=3):

    query_embedding = model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k
    )

    retrieved_docs = []

    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        retrieved_docs.append({
            "document": doc,
            "distance": dist  # lower = more similar
        })

    return retrieved_docs

In [ ]:
from groq import Groq

client = Groq(api_key="your_api_key")

In [10]:
def generate_root_cause(context):
    SYSTEM_PROMPT = """You are an expert in Hadoop Distributed File System (HDFS) operations and log analysis.

You will be given retrieved anomaly cases. Each case has five sections:
- Details: raw event data for the block
- Summary: what type of anomaly occurred
- Root Cause: why it likely occurred
- Important: severity and impact notes

Use these sections as evidence. Be specific — reference the actual event sequences and counts."""

    USER_PROMPT = f"""The following anomaly cases were retrieved as similar to the query:

{context}

Based on the evidence above, provide:
1. Most likely root cause (cite specific events or patterns from the cases)
2. Anomaly category: system / network / configuration
3. Confidence level: high / medium / low — and why
4. Mitigation steps (ordered by priority)"""

    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_PROMPT}
            ]
        )
        return response.choices[0].message.content

    except Exception as e:
        return f"[ERROR] Groq API call failed: {e}"

Step 6: End-to-End Pipeline Integration & Testing

In [11]:
#query = "Repeated block reception with high latency"
query = """
Repeated block reception events with missing acknowledgment and high latency
"""
#query = documents[0]

similar_docs = retrieve_similar_anomalies(query)

context = "\n\n".join([
    f"Case {i+1} (distance: {r['distance']:.2f}):\n{r['document']}"
    for i, r in enumerate(similar_docs)
])

In [12]:
result = generate_root_cause(context)

print(result)

Based on the evidence provided, here's my analysis:

1. Most likely root cause:
After examining the cases, I notice a recurring pattern of high-latency block operations with excessive retries of `E5` (Receiving block) and `E9` (PacketResponder for block terminating) events. This suggests network congestion between DataNodes causing pipeline stalls, which is a possible cause of the high latency. Specifically, I would point to the following events:
* `E5 -> E22 -> E5 -> E5`: A repeated pattern of receiving blocks, which indicates that the DataNode is experiencing issues processing incoming blocks.
* `E5 -> E9 -> E11 -> E9 -> E11 -> E9`: A pattern of retries of `E9` (PacketResponder for block terminating) events, suggesting that the data being sent is being dropped or not being processed correctly by the DataNode.
* `BLOCK* NameSystem[*]addStoredBlock: Redundant addStoredBlock request received for[*]on[*]size[*]`: A single event where a redundant request is received, which could be a symp